# Iteración, recursos y concurrencia

**Unidad 1 · Sesión 3**

Generadores, context managers y concurrencia con `asyncio`.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Generadores y consumo bajo demanda |
| 2 | `with`, decoradores y `@contextmanager` |
| 3 | `async`/`await` y tareas concurrentes |
| 4 | Tiempos límite y limpieza |

Al terminar, podrás reconocer cuándo conviene procesar valores uno a uno,
mantener un recurso abierto solo durante su uso y solapar esperas independientes.

## 1. Generadores y consumo bajo demanda

Una lista conserva sus elementos. Un **iterador** mantiene la posición de un
recorrido; `next()` solicita el siguiente valor. Cuando se agota, no vuelve al
inicio. Una función con `yield` crea un generador: produce un valor cuando el
consumidor lo pide y continúa desde donde se quedó.

In [18]:
values = iter([18.0, 27.0, 32.0])
print(next(values))
print(list(values))
print(list(values))  # The iterator is exhausted.

18.0
[27.0, 32.0]
[]


In [19]:
from collections.abc import Iterator


def readings(values: list[float]) -> Iterator[float]:
    for value in values:
        print(f"Reading {value}")
        yield value


stream = readings([18.0, 27.0, 32.0])
print("Generator created")
print(next(stream))
print(list(stream))

Generator created
Reading 18.0
18.0
Reading 27.0
Reading 32.0
[27.0, 32.0]


Crear `stream` no ejecutó el cuerpo de `readings`. El primer `next()` produjo
un solo valor; `list(stream)` consumió el resto. Esto permite recorrer datos
sin construir antes una lista con todos los resultados. Si el consumidor usa
`list()`, entonces sí materializa lo que queda.

### Ejercicio 1 · Números pares bajo demanda

Escribe `even_numbers(stop: int)`, una función generadora que produzca los
números pares desde cero hasta antes de `stop`. Comprueba que
`list(even_numbers(7))` sea `[0, 2, 4, 6]` y que `list(even_numbers(0))`
sea una lista vacía. Usa `yield`; no construyas una lista dentro de la función.

In [20]:
# Define even_numbers and check both cases.

from collections.abc import Iterator


def even_numbers(stop: int) -> Iterator[int]:
    '''Produce los números pares desde 0 hasta antes de stop, uno por uno.'''
    # Empiezo en 0 y avanzo de 2 en 2, así solo paso por los pares.
    numero = 0
    # Mientras el número sea menor que stop, lo entrego con yield.
    # "Antes de stop" significa que stop no se incluye, por eso uso < y no <=.
    while numero < stop:
        # yield entrega el número y la función se pausa aquí
        # hasta que alguien pida el siguiente.
        yield numero
        numero += 2


# Comprobación 1: con stop = 7, los pares antes de 7 son 0, 2, 4 y 6.
assert list(even_numbers(7)) == [0, 2, 4, 6]

# Comprobación 2: con stop = 0 no hay ningún par antes de 0, así que la lista queda vacía.
assert list(even_numbers(0)) == []

# Compruebo que de verdad entrega los valores uno por uno con next().
pares = even_numbers(7)
print("Primero:", next(pares))
print("Segundo:", next(pares))
print("Lo que queda:", list(pares))

# Explicación:
# No construí ninguna lista dentro de la función. Con yield, la función
# entrega un número, se pausa, y continúa desde ahí cuando le piden el siguiente.
# La lista solo se crea afuera, cuando yo uso list() para comprobar el resultado.
# Esto sirve cuando hay muchos datos, por ejemplo, miles de lecturas de un sensor:
# no necesito tenerlas todas en memoria, las proceso una por una.

Primero: 0
Segundo: 2
Lo que queda: [4, 6]


## 2. Recursos con `with`

Un archivo debe cerrarse al terminar de usarlo. `with` delimita ese periodo:
al salir del bloque, Python llama al cierre del recurso, también si dentro
ocurre un error. El archivo se puede recorrer línea por línea.

In [21]:
from pathlib import Path
from tempfile import TemporaryDirectory


with TemporaryDirectory() as directory:
    path = Path(directory) / "readings.txt"
    path.write_text("north;18.0\nsouth;27.0\n", encoding="utf-8")

    with path.open(encoding="utf-8") as file:
        first_line = next(file).strip()
        print(first_line)

    assert file.closed

north;18.0


### Salida por error

El cierre no depende de que el bloque termine normalmente. El error puede
atenderse fuera de `with` y el archivo ya estará cerrado.

In [22]:
with TemporaryDirectory() as directory:
    path = Path(directory) / "readings.txt"
    path.write_text("invalid\n", encoding="utf-8")

    try:
        with path.open(encoding="utf-8") as file:
            raise ValueError("Invalid reading")
    except ValueError as error:
        print(error)

    assert file.closed

Invalid reading


### Qué es un decorador

Un decorador recibe una función y devuelve otra que añade comportamiento.
La forma `@announce` aplica `announce` a la función definida debajo. Aquí
solo observamos esa idea; no hace falta crear una biblioteca de decoradores.


In [23]:
def announce(function):
    def wrapper():
        print("Starting")
        return function()

    return wrapper


@announce
def show_message():
    print("Reading complete")


show_message()

Starting
Reading complete


### Crear un context manager con `@contextmanager`

`@contextmanager` permite escribir un context manager sin definir una clase.
La función llega hasta `yield` al entrar en `with`; el bloque `finally` se
ejecuta al salir. En esta demostración, el recurso es un archivo en memoria.

In [24]:
from contextlib import contextmanager
from io import StringIO


@contextmanager
def open_text(text: str) -> Iterator[StringIO]:
    file = StringIO(text)
    try:
        yield file
    finally:
        file.close()


with open_text("north\nsouth\n") as file:
    print(next(file).strip())

assert file.closed

north


### Ejercicio 2 · Cierre de un archivo

Crea un archivo temporal con dos líneas. Ábrelo con `with`, lee solo la primera
y comprueba que esté cerrado después del bloque. Luego provoca un `ValueError`
dentro de otro bloque `with`, atiéndelo fuera del bloque y comprueba de nuevo
que el archivo quedó cerrado.

In [25]:
# Use TemporaryDirectory and Path; check file.closed in both situations.

from pathlib import Path
from tempfile import TemporaryDirectory

# TemporaryDirectory crea una carpeta temporal que se borra sola al terminar.
# Así no dejo archivos de prueba en Colab.
with TemporaryDirectory() as carpeta:
    # Armo la ruta del archivo dentro de la carpeta temporal.
    ruta = Path(carpeta) / "monitoreo.txt"

    # Escribo dos líneas: pozo y pH. El \n separa las líneas.
    ruta.write_text("pozo_1;7.2\npozo_2;6.8\n", encoding="utf-8")

    # Parte 1: abro el archivo con with y leo solo la primera línea.
    with ruta.open(encoding="utf-8") as archivo:
        # next() me da solo la siguiente línea, que es la primera.
        # strip() le quita el salto de línea del final.
        primera_linea = next(archivo).strip()
        print("Primera línea:", primera_linea)

        # Dentro del with, el archivo todavía está abierto.
        assert not archivo.closed

    # Ya salí del with, así que el archivo debe estar cerrado.
    assert archivo.closed
    print("¿Cerrado después del primer with?", archivo.closed)
    assert primera_linea == "pozo_1;7.2"

    # Parte 2: provoco un error dentro de otro with.
    # El try está AFUERA del with, para atender el error fuera del bloque.
    try:
        with ruta.open(encoding="utf-8") as archivo_con_error:
            linea = next(archivo_con_error).strip()
            # Simulo que encontré una lectura inválida y lanzo un ValueError.
            raise ValueError(f"Lectura inválida: {linea}")
    except ValueError as error:
        print("Error atendido fuera del with:", error)

    # Aunque el bloque terminó por un error, with cerró el archivo de todas formas.
    assert archivo_con_error.closed
    print("¿Cerrado después del error?", archivo_con_error.closed)

# Explicación:
# with garantiza que el archivo se cierre al salir del bloque, termine bien o con error.
# En la parte 1 salí normalmente y el archivo quedó cerrado.
# En la parte 2 salí por un ValueError, y aun así el archivo quedó cerrado.
# Por eso no necesito escribir archivo.close() a mano ni preocuparme
# de que un error deje el archivo abierto.
# Es como un permiso de trabajo con cierre obligatorio: se cierra siempre,
# aunque el trabajo se interrumpa por un incidente.

Primera línea: pozo_1;7.2
¿Cerrado después del primer with? True
Error atendido fuera del with: Lectura inválida: pozo_1;7.2
¿Cerrado después del error? True


## 3. Concurrencia con `asyncio`

Primero simulamos una lectura **síncrona**: `time.sleep` bloquea el hilo durante
los segundos indicados. Dos llamadas consecutivas de 3 y 2 segundos tardan
aproximadamente 5 segundos. Los mensajes permiten observar cuándo comienza y
termina cada operación.

In [26]:
import asyncio
import time
from time import perf_counter


def fetch_reading_sync(station: str, delay: float) -> tuple[str, float]:
    values = {"north": 18.0, "south": 27.0, "west": 32.0}
    print(f"Requesting {station}")
    time.sleep(delay)
    print(f"Received {station}")
    return station, values[station]


start = perf_counter()
print(fetch_reading_sync("north", 3))
print(fetch_reading_sync("south", 2))
print(f"Synchronous: {perf_counter() - start:.1f} s")

Requesting north
Received north
('north', 18.0)
Requesting south
Received south
('south', 27.0)
Synchronous: 5.0 s


### La versión asíncrona

`async def` define una corrutina. `await asyncio.sleep()` suspende esa corrutina
y permite que otras tareas avancen. Cambiar solo `def` por `async def` no haría
que `time.sleep` dejara de bloquear: también necesitamos una espera compatible.

Dos `await` consecutivos siguen siendo **secuenciales**. En Jupyter y Colab usamos
`await` directamente en una celda.

In [27]:
async def fetch_reading(station: str, delay: float) -> tuple[str, float]:
    values = {"north": 18.0, "south": 27.0, "west": 32.0}
    print(f"Requesting {station}")
    await asyncio.sleep(delay)
    print(f"Received {station}")
    return station, values[station]


start = perf_counter()
north = await fetch_reading("north", 3)
south = await fetch_reading("south", 2)
print(north, south, f"Sequential: {perf_counter() - start:.1f} s")

Requesting north
Received north
Requesting south
Received south
('north', 18.0) ('south', 27.0) Sequential: 5.0 s


### Adaptar una función síncrona con `to_thread`

Si una biblioteca de E/S solo ofrece una función síncrona, `asyncio.to_thread`
puede ejecutarla en otro hilo. Este wrapper conserva la función original y
permite esperar su resultado con `await`. Pasamos la función **sin llamarla**,
seguida de sus argumentos. Un único `await` no hace que la lectura dure menos.

Esto sirve para esperas bloqueantes de E/S. No garantiza acelerar cálculos de
Python, y cancelar la espera no detiene automáticamente un hilo que ya trabaja.

In [28]:
async def fetch_reading_in_thread(station: str, delay: float) -> tuple[str, float]:
    return await asyncio.to_thread(fetch_reading_sync, station, delay)


print(await fetch_reading_in_thread("north", 2))

Requesting north
Received north
('north', 18.0)


### Una tarea en segundo plano con `create_task`

`create_task()` programa una corrutina para que avance mientras la corrutina
principal hace otro trabajo y cede el control. Conservamos la tarea en una
variable y al final usamos `await` para obtener su resultado y observar errores.
Aquí la lectura continúa durante la pausa de un segundo de la tarea principal.
No es un trabajo persistente: depende de que el event loop siga activo.

In [29]:
reading_task = asyncio.create_task(fetch_reading("north", 3))
print("Main task continues")
await asyncio.sleep(1)
print("Main task is ready to use the reading")
print(await reading_task)

Main task continues
Requesting north
Main task is ready to use the reading
Received north
('north', 18.0)


### Reunir resultados con `gather`

`gather()` programa las corrutinas y devuelve los resultados en el orden en que
las pasamos, aunque terminen en otro orden. Las esperas de 3 y 2 segundos se
solapan: el total se acerca a 3 segundos.

In [30]:
start = perf_counter()
north, south = await asyncio.gather(
    fetch_reading("north", 3),
    fetch_reading("south", 2),
)
print(north, south, f"Gather: {perf_counter() - start:.1f} s")

Requesting north
Requesting south
Received south
Received north
('north', 18.0) ('south', 27.0) Gather: 3.0 s


### Delimitar las tareas con `TaskGroup`

`TaskGroup` espera sus tareas al salir de `async with`. Conservamos cada objeto
tarea para consultar después su resultado. Si una tarea falla con una excepción
que no es de cancelación, el grupo cancela las restantes y espera su limpieza.

En cambio, `gather` con sus opciones predeterminadas propaga la primera excepción,
pero las otras tareas pueden continuar. Aquí ambos ejemplos tienen éxito; la
diferencia está en cómo organizan su finalización y sus errores.

In [31]:
start = perf_counter()
async with asyncio.TaskGroup() as group:
    north_task = group.create_task(fetch_reading("north", 3))
    south_task = group.create_task(fetch_reading("south", 2))
print(north_task.result(), south_task.result())
print(f"TaskGroup: {perf_counter() - start:.1f} s")

Requesting north
Requesting south
Received south
Received north
('north', 18.0) ('south', 27.0)
TaskGroup: 3.0 s


Los tiempos son aproximados. Observa que ambas solicitudes empiezan antes de
que llegue la primera respuesta. La concurrencia permite solapar esperas
independientes; no acelera por sí sola un cálculo intensivo.

### Ejercicio 3 · Dos lecturas concurrentes

Crea dos tareas con `TaskGroup`: `fetch_reading("north", 3)` y
`fetch_reading("west", 1)`. Después del bloque, comprueba con `assert`
los dos resultados. Explica por qué `west` puede anunciarse primero aunque
la tarea de `north` se haya creado antes.

In [32]:
# Create two tasks, read their results after TaskGroup, and check both tuples.

# Mido el tiempo para comprobar que las dos lecturas se hicieron al mismo tiempo.
inicio = perf_counter()

# TaskGroup lanza las dos tareas juntas y espera a que las dos terminen
# antes de salir del bloque async with.
async with asyncio.TaskGroup() as grupo:
    # Creo primero la tarea de north (tarda 3 segundos)
    # y después la de west (tarda 1 segundo).
    tarea_north = grupo.create_task(fetch_reading("north", 3))
    tarea_west = grupo.create_task(fetch_reading("west", 1))

# Ya salí del bloque, así que las dos tareas terminaron.
# Con .result() obtengo lo que devolvió cada una.
resultado_north = tarea_north.result()
resultado_west = tarea_west.result()
duracion = perf_counter() - inicio

print("north:", resultado_north)
print("west:", resultado_west)
print(f"Tiempo total: {duracion:.1f} s")

# Compruebo que cada tarea devolvió la estación y su valor correctos.
assert resultado_north == ("north", 18.0)
assert resultado_west == ("west", 32.0)

# Compruebo que tardó cerca de 3 segundos (la más lenta) y no 4 (3 + 1).
# Eso demuestra que las esperas se hicieron al mismo tiempo.
assert duracion < 4

# Explicación:
# west se anuncia primero ("Received west") aunque north se creó antes,
# porque las dos tareas empiezan casi al mismo tiempo y west solo espera 1 segundo,
# mientras que north espera 3. El orden en que terminan depende de cuánto
# esperan, no del orden en que se crearon.
# Es como mandar dos muestras a dos laboratorios el mismo día:
# llega primero el resultado del laboratorio más rápido,
# aunque esa muestra se haya enviado en segundo lugar.
# Aun así, cada resultado queda guardado en su propia tarea,
# así que no se mezclan: tarea_north siempre tiene el de north.

Requesting north
Requesting west
Received west
Received north
north: ('north', 18.0)
west: ('west', 32.0)
Tiempo total: 3.0 s


## 4. Tiempo límite y limpieza

Una operación puede tardar demasiado. `asyncio.timeout()` establece un tiempo
máximo para el bloque. Si se agota, se produce `TimeoutError`. La corrutina
puede usar `finally` para limpiar su estado incluso cuando se cancela la espera.

In [33]:
events: list[str] = []


async def wait_for_update() -> None:
    events.append("started")
    try:
        await asyncio.Event().wait()
    finally:
        events.append("cleaned up")


try:
    async with asyncio.timeout(2):
        await wait_for_update()
except TimeoutError:
    print("Deadline reached")

assert events == ["started", "cleaned up"]
print(events)

Deadline reached
['started', 'cleaned up']


### Ejercicio 4 · Comprobar la limpieza

Escribe una corrutina que añada `"started"` a una lista, espere 3 segundos
y añada `"cleaned up"` en `finally`. Llámala dentro de
`asyncio.timeout(1)`, atiende `TimeoutError` y comprueba que ambos mensajes
quedaron en la lista. Después prueba una espera de 1 segundo con un límite de 3 segundos.

In [34]:
# Define a coroutine with try/finally; check a timeout and a completed wait.

async def lectura_con_limpieza(eventos: list[str], segundos: float) -> None:
    '''Registra el inicio, espera los segundos indicados y siempre registra la limpieza.'''
    eventos.append("started")
    try:
        # Simulo una lectura que tarda cierto tiempo.
        await asyncio.sleep(segundos)
    finally:
        # finally se ejecuta siempre: si la espera terminó bien
        # o si se canceló porque se acabó el tiempo límite.
        eventos.append("cleaned up")


# Caso 1: la lectura tarda 3 segundos, pero el límite es de 1 segundo.
# Le paso una lista nueva para que los eventos de cada caso no se mezclen.
eventos_caso_1: list[str] = []
hubo_timeout = False

# El try está afuera del async with para atender el error fuera del bloque.
try:
    async with asyncio.timeout(1):
        await lectura_con_limpieza(eventos_caso_1, 3)
except TimeoutError:
    hubo_timeout = True
    print("Caso 1: se agotó el tiempo límite")

print("Eventos caso 1:", eventos_caso_1)

# Compruebo que sí se agotó el tiempo.
assert hubo_timeout

# Compruebo que, aunque se canceló, la limpieza sí se registró.
assert eventos_caso_1 == ["started", "cleaned up"]


# Caso 2: la lectura tarda 1 segundo y el límite es de 3 segundos.
# Esta vez sí debe terminar a tiempo.
eventos_caso_2: list[str] = []
hubo_timeout = False

try:
    async with asyncio.timeout(3):
        await lectura_con_limpieza(eventos_caso_2, 1)
    print("Caso 2: la espera terminó a tiempo")
except TimeoutError:
    hubo_timeout = True

print("Eventos caso 2:", eventos_caso_2)

# Compruebo que no se agotó el tiempo.
assert not hubo_timeout

# Compruebo que también se registraron los dos eventos.
assert eventos_caso_2 == ["started", "cleaned up"]

# Explicación:
# En el caso 1 la espera se canceló al cumplirse 1 segundo, pero finally
# se ejecutó de todas formas y registró "cleaned up".
# En el caso 2 la espera terminó normalmente y finally también se ejecutó.
# La lista queda igual en los dos casos; lo que cambia es si hubo TimeoutError.
# Por eso finally es el lugar correcto para la limpieza: se ejecuta siempre.
# Es como el procedimiento de salida de un espacio confinado:
# se hace igual si el trabajo terminó o si se suspendió por tiempo.

Caso 1: se agotó el tiempo límite
Eventos caso 1: ['started', 'cleaned up']
Caso 2: la espera terminó a tiempo
Eventos caso 2: ['started', 'cleaned up']


## Cierre

- Un generador produce valores cuando se solicitan; consumirlo por completo
  puede volver a ocupar memoria para guardar los resultados.
- `with` delimita el uso de un recurso; `@contextmanager` permite definir
  esa entrada y salida mediante una función.
- `asyncio` ayuda a solapar esperas independientes. Usa un tiempo límite
  cuando una espera no deba continuar indefinidamente.

En un archivo `.py` se inicia el event loop desde una función principal con
`asyncio.run(main())`; en esta notebook usamos `await` directamente.

## Referencias

- [Tutorial de Python: generadores](https://docs.python.org/3/tutorial/classes.html#generators).
- [Python: `contextlib`](https://docs.python.org/3/library/contextlib.html).
- [Python: corrutinas y tareas](https://docs.python.org/3/library/asyncio-task.html).
- [Python: ejecutar `asyncio`](https://docs.python.org/3/library/asyncio-runner.html).